# UAE Micro-Mobility Compliance & Safety HUD Engine

## End-to-End Pipeline on Google Colab T4 GPU

This notebook runs the complete compliance pipeline on a free Colab T4 GPU instance:

1. **Environment Setup** - Install dependencies
2. **Test Data Generation** - Synthesize a 30-second commute video + GPS log
3. **Pipeline Execution** - Run `main.py` with full telemetry
4. **Visual Verification** - Preview the HUD output and audit trail

In [ ]:
# Cell 1: Environment Setup

# Install dependencies on Colab. Uses a T4 GPU (Runtime -> Change runtime type -> T4 GPU).

import os, sys, subprocess

# Detect if we are on Colab
IN_COLAB = 'google.colab' in sys.modules
print(f'Running on Colab: {IN_COLAB}')

if IN_COLAB:
    # Core dependencies
    !pip install -q \
        ultralytics>=8.0 \
        opencv-python-headless>=4.8 \
        pydantic-settings>=2.0 \
        opentelemetry-api>=1.20 \
        opentelemetry-sdk>=1.20 \
        opentelemetry-exporter-otlp-proto-http>=1.20 \
        opentelemetry-semantic-conventions>=0.40b0 \
        opentelemetry-proto>=1.20 \
        requests>=2.31 \
        numpy>=1.24

    # VLM + fine-tuning dependencies (transformers + peft + bitsandbytes)
    !pip install -q \
        transformers>=4.46 \
        peft>=0.10 \
        bitsandbytes>=0.43 \
        Pillow>=10.0

    # Unsloth for efficient QLoRA fine-tuning
    !pip install -q --no-deps 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'

    # Restart kernel to pick up new packages
    print('Dependencies installed. Restarting runtime...')
    os._exit(0)

print('Environment ready.')

In [ ]:
# Cell 2: Test Data Generation

# Generate a 30-second commute video (30 FPS = 900 frames) and a GPS log.

import cv2
import numpy as np
import json
import os

# Check CUDA availability for the VLM
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

FPS = 30
DURATION = 30  # seconds
N_FRAMES = FPS * DURATION
WIDTH, HEIGHT = 640, 480

# Create output directories
os.makedirs('demo_data', exist_ok=True)

# --- Generate synthetic commute video ---
# Simulates an e-scooter moving along a road with surface changes.
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('/tmp/demo_data/commute_raw.mp4', fourcc, FPS, (WIDTH, HEIGHT))

# Track path (simulates GPS movement: ~15 km/h scooter)
gps_points = []
lat_start, lon_start = 25.0755, 55.1384  # Dubai Marina area

# Pre-calculate movement per frame (~15 km/h = 4.17 m/s)
meters_per_frame = 4.17 / FPS  # ~0.14 m per frame
# Convert to approx lat/lon delta (1 deg lat ~111km, 1 deg lon ~90km at this latitude)
lat_delta = meters_per_frame / 111000
lon_delta = meters_per_frame / 90000

for i in range(N_FRAMES):
    # Base background: light asphalt texture
    frame = np.ones((HEIGHT, WIDTH, 3), dtype=np.uint8) * np.array([120, 120, 120])

    # Add some visual noise for realism
    noise = np.random.randint(-10, 10, (HEIGHT, WIDTH, 3), dtype=np.int16)
    frame = np.clip(frame.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    # Road / surface simulation: change surface every 10 seconds
    elapsed = i / FPS
    if (elapsed // 10) % 2 == 0:
        # Red track surface - draw red lines on road
        surface_color = (0, 0, 200)  # BGR red
        road_color = (80, 80, 80)
    else:
        # Grey sidewalk - draw grey pavement texture
        surface_color = (100, 100, 100)  # BGR grey
        road_color = (100, 100, 100)

    # Draw road surface
    cv2.rectangle(frame, (0, HEIGHT // 2 - 40), (WIDTH, HEIGHT), road_color, -1)

    # Draw surface stripes
    for j in range(0, WIDTH, 30):
        cv2.line(frame, (j, HEIGHT // 2), (j + 15, HEIGHT // 2), surface_color, 3)

    # --- Scooter position (moves left to right) ---
    scooter_x = int(50 + (i / N_FRAMES) * (WIDTH - 100))
    scooter_y = HEIGHT // 2 + 20

    # Draw scooter
    cv2.rectangle(frame, (scooter_x, scooter_y), (scooter_x + 40, scooter_y + 20), (200, 200, 200), -1)
    # Draw rider (circle)
    cv2.circle(frame, (scooter_x + 20, scooter_y - 10), 8, (180, 180, 180), -1)
    # Draw helmet (green circle)
    cv2.circle(frame, (scooter_x + 20, scooter_y - 15), 5, (0, 255, 0), -1)

    # Every 15 seconds, add a crosswalk
    if 14 < elapsed < 16 or 29 < elapsed < 31:
        cv2.line(frame, (scooter_x - 20, scooter_y + 30), (scooter_x + 60, scooter_y + 30), (255, 255, 255), 2)
        cv2.line(frame, (scooter_x - 20, scooter_y + 35), (scooter_x + 60, scooter_y + 35), (255, 255, 255), 2)

    out.write(frame)

    # Record GPS point
    gps_points.append({
        'timestamp': round(elapsed, 3),
        'lat': round(lat_start + lat_delta * i, 6),
        'lon': round(lon_start + lon_delta * i, 6),
        'speed_kmh': 15.0
    })

out.release()
print(f'Generated video: {N_FRAMES} frames, {DURATION}s at {FPS} FPS')

# --- Generate GPS log ---
with open('/tmp/demo_data/gps_log.json', 'w') as f:
    json.dump(gps_points, f, indent=2)
print(f'Generated GPS log: {len(gps_points)} points')

print('Test data ready: /tmp/demo_data/commute_raw.mp4, /tmp/demo_data/gps_log.json')

In [ ]:
# Cell 3: Pipeline Execution

# Clone the repo and run the pipeline with full telemetry.

import os, sys

# Clone the repository (if not already present)
if not os.path.exists('.git'):
    !git clone https://github.com/ahmedaw-official/emirates-hud.git .

# Install the package in development mode
!pip install -q -e . 2>&1 | tail -3

# Run the pipeline on the GPU with mock VLM (safe default for Colab)
from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

%time python main.py \
    --input-video /tmp/demo_data/commute_raw.mp4 \
    --output-video /tmp/demo_data/commute_hud_demo.mp4 \
    --gps-log /tmp/demo_data/gps_log.json \
    --audit-log /tmp/demo_data/audit_trail.json \
    --mock-vlm 2>&1 | tail -20

print('Pipeline execution complete.')
print(f'Output video: /tmp/demo_data/commute_hud_demo.mp4')
print(f'Audit log: /tmp/demo_data/audit_trail.json')

In [ ]:
# Cell 4: Visual Verification

# Extract a 5-second slice from the output video and render an animated GIF.
import cv2
import numpy as np
import json
from IPython.display import Image, display
import subprocess

OUTPUT_VIDEO = '/tmp/demo_data/commute_hud_demo.mp4'
SLICE_START = 10  # seconds
SLICE_DURATION = 5  # seconds
FPS = 30

# Extract frames from the 5-second window
cap = cv2.VideoCapture(OUTPUT_VIDEO)
start_frame = SLICE_START * FPS
end_frame = start_frame + SLICE_DURATION * FPS
cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

frames = []
idx = start_frame
while idx < end_frame:
    ret, frame = cap.read()
    if not ret:
        break
    # Resize for display
    frame_small = cv2.resize(frame, (480, 360))
    frames.append(frame_small)
    idx += 1
cap.release()

print(f'Extracted {len(frames)} frames for the {SLICE_DURATION}s slice')

# Build animated GIF using ffmpeg
gif_path = '/tmp/demo_data/commute_preview.gif'
frames_dir = '/tmp/demo_data/frames'
os.makedirs(frames_dir, exist_ok=True)

for i, f in enumerate(frames):
    cv2.imwrite(f'{frames_dir}/frame_{i:03d}.png', f)

# Use ffmpeg to create the GIF (Colab has ffmpeg pre-installed)
subprocess.run([
    'ffmpeg', '-y', '-framerate', str(FPS // 3),  # 10 FPS for the GIF
    '-i', f'{frames_dir}/frame_%03d.png',
    '-vf', 'scale=480:-1:flags=lanczos',
    '-loop', '0',
    gif_path
], capture_output=True)

# Display the animated GIF
display(Image(filename=gif_path))

# Display audit trail summary
with open('/tmp/demo_data/audit_trail.json', 'r') as f:
    audit = json.load(f)

summary = audit['summary']
print('\n=== PIPELINE SUMMARY ===')
print(f"Total frames processed:  {summary['total_frames_processed']}")
print(f"VLM triggers:            {summary['vlm_triggers_count']}")
print(f"Total fines accrued:     {summary['total_fines_accrued_aed']} AED")
print(f"Violations detected:     {summary['violations_detected_count']}")
print(f"Processing errors:       {summary['processing_errors']}")

# Show first 5 and last 5 entries as examples
entries = audit['entries']
print(f'\n=== FIRST 5 FRAME STATES ===')
for e in entries[:5]:
    print(f"Frame {e['frame_id']}: state={e['state']}, fine={e['fine_risk_aed']} AED, "
          f"vlm_triggered={e['vlm_triggered']}, reason={e['trigger_reason']}")

print(f'\n=== LAST 5 FRAME STATES ===')
for e in entries[-5:]:
    print(f"Frame {e['frame_id']}: state={e['state']}, fine={e['fine_risk_aed']} AED, "
          f"vlm_triggered={e['vlm_triggered']}, reason={e['trigger_reason']}")